# Email Agent — Human-in-the-Loop (Module 3 finale)

> The capstone that turns every module-3 concept — **dynamic tools, dynamic prompts, state & context schemas, human-in-the-loop** — into a single working email assistant.

## Why we're building this

Most chatbots **talk**. This agent **acts**: it authenticates you against a mailbox, reads the inbox, prepares a reply, and sends it. Acting has consequences (wrong recipient, wrong tone, wrong legal claim), so the golden rule is:

> **Read = automatic · Draft = automatic · Send = only after YOU approve (approve / reject / edit).**

This notebook is the teaching version of `3.5_email_agent.py` (the server the chat UI talks to). Same logic, sandboxed in a notebook so we can inspect every step.

## The full schema (what it does)

```mermaid
flowchart LR
    U[You: chat message] --> AUTH[authenticate tool]
    AUTH -- "Command(update)" --> S[state: authenticated=True]
    S --> GATE{authenticated?}
    AUTH --> GATE
    GATE -- No --> AUTH
    GATE -- Yes --> INBOX[check_inbox tool]
    INBOX --> DRAFT[Model drafts a reply]
    DRAFT --> SEND[send_email tool]
    SEND -. "pause before running" .-> HITL[Human approval]
    HITL -- "approve / reject / edit" --> RUN[Tool runs or is blocked]
    RUN --> DONE[Done: report back to you]
```

## Step-by-step (the sequence, played out)

```mermaid
sequenceDiagram
    participant You
    participant Agent
    participant Mailbox
    You->>Agent: "julie@example.com, password123"
    Agent->>Agent: authenticate() -- the ONLY tool before login
    Agent-->>You: (state flipped, prompt + tools unlock)
    Agent->>Mailbox: check_inbox()
    Agent->>Agent: draft a friendly reply
    Agent->>Agent: tries send_email -> INTERRUPT
    Note over You,Agent: __interrupt__ shows to / subject / body
    You->>Agent: resume: approve (or reject / edit)
    Agent->>Mailbox: send_email runs
    Agent-->>You: "Email sent to ..."
```

## Outline of what each cell proves
- **authenticate** → only then are the inbox tools unlocked (dynamic tools + dynamic prompt driven by state)
- **check_inbox** → reads the (fake) mailbox that lives inside a tool
- **send_email** → pauses for a human decision before it ever runs


## 1 · Setup — silence noise, load secrets

**Why:**
- `warnings.filterwarnings` hides the Pydantic serializer warnings that pollute long agent runs.
- `load_dotenv()` reads `.env` at the repo root, giving us `GOOGLE_API_KEY` for the free Gemini model.
**What you'll see:** nothing printed — just a clean environment for the whole run.

In [1]:
# 1 · Silence Pydantic noise, load GOOGLE_API_KEY from .env

import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

from dotenv import load_dotenv

load_dotenv()

True

## 2 · The model — one shared object

**Why:**
- `create_agent` needs a chat-model *instance*. We build it once here (free `gemini-3.1-flash-lite`) and reuse it everywhere.
- The API key comes from the environment loaded in cell 1 — order matters.
**What you'll see:** the object exists silently; nothing is sent to Gemini until we invoke the agent.

In [2]:
# 2 · One shared Gemini model object, reused by every agent call

import os

from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gemini-3.1-flash-lite",
    model_provider="google-genai",
    api_key=os.getenv("GOOGLE_API_KEY"),
)


## 3 · Context schema — what the SYSTEM knows

**Why:**
- `EmailContext` is the `context_schema`: fixed identity + credentials the **app injects** at call time (`context=EmailContext()`).
- It is *read-only*: tools reach it through `runtime.context`, and the model can never modify it.
**Key distinction:** this is *injected and immutable* — next cell is what the agent *builds up*.

In [3]:
# 3 · EmailContext: credentials the APP injects (context_schema) -- immutable

from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

## 4 · State schema — what the AGENT remembers

**Why:**
- `AuthenticatedState` extends `AgentState` (which already holds `messages`) with one custom flag: `authenticated: bool`.
- This flag is what every dynamic decision (tools + prompt) will read. It persists across turns via the checkpointer.
**Key distinction:** `context` = injected, immutable; `state` = built up by the agent during the conversation.

In [4]:
# 4 · AuthenticatedState: what the AGENT remembers (state_schema) -- built up at runtime

from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

## 5 · The 3 tools — the only way the model touches the world

**Why:**
- `@tool` turns each function into a callable the model can request by name + JSON args.
- The **docstring becomes the schema the model sees** — it decides when to call based on that text, so keep docstrings short (re-sent every turn).

Design split:
| tool | effect | changes state? |
|---|---|---|
| `check_inbox` | returns the mailbox | no |
| `send_email` | ‘sends’ (fake) | no — but gets paused later |
| `authenticate` | flips `authenticated` | yes — `Command(update=...)` |

Only `authenticate` uses `Command(update)` because it's the only one that must **change the graph's state**, not just return a string.

In [5]:
# 5 · The world's only interface: read / act / authenticate (only this one changes state)

from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

## 6 · Dynamic tool gating — security by absence

**Why:**
- `wrap_model_call` lets us change the toolset *per model call*, based on state.
- **Before login:** the model only sees `authenticate`. It literally cannot call inbox tools.
- **After login:** it sees `check_inbox` + `send_email`.

Hiding a tool is absolute; telling the model 'don't' is just a suggestion — absence beats prompting.

In [6]:
# 6 · Dynamic tools: model sees ONLY what authentication allows

from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

>Note: the prompts were modified since filming to constrain the model to more reliably match the filmed sequence. You may still experience different responses from the model, which is expected. You may need to modify the human message to provide appropriate responses.

## 7 · Dynamic prompt — persona follows authentication

**Why:**
- `dynamic_prompt` swaps the system prompt per call: before login an *authenticator*, after login an *email assistant* that checks the inbox first.
- Keeps the model focused on what it's *allowed* to do right now.

In [7]:
# 7 · Dynamic prompt: persona follows the authenticated flag

from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails. 
Your first step after authentication is to check the inbox."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

## 8 · Assemble the agent

**Why each argument:**
- `model` — our shared Gemini instance (cell 2).
- `tools` — all three, but dynamic gating (cell 6) controls which are visible.
- `checkpointer=InMemorySaver()` — **required for HITL** (the graph must park mid-run) and gives thread memory.
- `state_schema` / `context_schema` — cells 3 & 4.
- `HumanInTheLoopMiddleware(interrupt_on={...})` — pause only `send_email`; `authenticate` and `check_inbox` run freely.

In [8]:
# 8 · Assemble agent: tools + state/context schemas + checkpointer + HITL (send = pause)

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model,
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )


## 9 · Turn 1 — authenticate and let it work

**Why:**
- The only tool available is `authenticate`, so the model *must* call it.
- State flips, tools unlock, it checks the inbox and drafts a reply.
- When it requests `send_email`, the middleware **interrupts** → `response` gains a `__interrupt__` key with the pending action.

In [9]:
# 9 · Turn 1: authenticate -> inbox -> draft -> pause at send_email

from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

[{'type': 'text', 'text': 'The inbox has been checked. You have one new email from Jane (jane@example.com) asking to grab coffee while she is in town next week. \n\nWould you like me to send a reply to Jane? If so, please let me know what you would like to say.', 'extras': {'signature': 'EoUBCoIBAWkUfRNVWfDQ0gD9xPss9I2oMgsC8adFSk712ZPCm7FEKsvlFM8NJLawwnZcJXOH3EI1uVUE6NCRSUfCpeCLiBiHzps8AgPk2T7rvP5RgNHUs/0EZWq7eUh/U2pXoVYxhu7wj6viU3mKMrf6NMqrFISLHtp2Ji/7O8K3SYQCMLZDfw=='}}]


## 10 · Turn 2 — trigger the send (same thread)

**Why:**
- Same `thread_id` = same conversation: the agent remembers what it drafted.
- "any draft is fine" → it calls `send_email` → interrupt fires again.
- `response['__interrupt__']` now holds the exact email that wants to go out.

In [10]:
# 10 · Turn 2 (same thread): tell it to send -> send_email INTERRUPTS


response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

[]


## 11 · Inspect the paused action

**Why:**
- This is the human-in-the-loop contract: before anything sends, we extract exactly what would go out (here: the email body).
- You (the human) get to see it — without this, approval is just a rubber stamp.

In [11]:
# 11 · Show the human exactly what would be sent

print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Jane,

That sounds like a great idea! I'd love to grab a coffee while you're in town next week. Let me know which day works best for you and we can coordinate.

Best,
Julie


## 12 · The human decision — approve / reject / edit

**Why:**
- `Command(resume=...)` unpauses the parked graph on the same thread.
- `approve` → the tool really runs. `reject` → blocked. `edit` → args swapped first.
- Only a decision on a *pending* action resumes anything — one pending interrupt per thread at a time (the 3.3 lesson).

In [12]:
# 12 · HUMAN DECISION: resume with approve (or reject / edit)

from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'The email has been sent to Jane.', 'extras': {'signature': 'EnMKcQFpFH0TsMKzYWfmDkFDFrV9B2vfBfq+I8XCXKGAuN9S4woKnsdDH7zAsxP38dTaPL2jwDeMI8SsIb3aAImIER4iZxS9OVEPLAML7pA5c9pfvUxLIgtCTHsfSMdvzBuP1Xs3fiL1wPX1k7cKBcUHtBzE'}}]


## 13 · Inspect the final state

**Why:**
- `pprint` shows the whole transcript: the ToolMessages proving `check_inbox` and `send_email` really ran, plus the model's final summary — the full proof the loop completed end-to-end.

In [13]:
# 13 · Inspect the final, complete state

from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='julie@example.com, password123', additional_kwargs={}, response_metadata={}, id='b89a58e2-7071-4bde-a75b-f802920938b6'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'authenticate', 'arguments': '{"email": "julie@example.com", "password": "password123"}'}, '__gemini_function_call_thought_signatures__': {'call_189833': 'EnMKcQFpFH0TmJE8nKWnsdPeGTwQ5BUUOyPiXOQOX/cAOZe6/KjTm+JHXcsNm9NJJA3neqCLjqNZXmwlV7yeODAAtdA6X3M182u0uxlsTafaai975U58T2dW8I0aD88vHSQvR4H/FKY5oqw/EIYJUJGiiXCV'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0dade-b9c1-70b0-a2b8-ce4cd4499303-0', tool_calls=[{'name': 'authenticate', 'args': {'email': 'julie@example.com', 'password': 'password123'}, 'id': 'call_189833', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 28, '